# Session 4 — Weights, Systematics, Fitting, and Limits

In the final session we treat the bb + MET selection as a **full analysis**: we include event weights and simple systematic uncertainties, perform a binned likelihood fit of signal + backgrounds to data, assess goodness of fit, and derive an upper limit on the signal strength. This mirrors, in a simplified way, the statistical treatment used in the CMS bb + MET JHEP result.

**Learning objectives:**
- Understand event weights and systematic uncertainties in a bb + MET analysis
- Perform a binned likelihood fit of signal + backgrounds to data in MET bins
- Assess goodness of fit (χ², p-values, pulls)
- Compute an upper limit on signal strength for the bbMET signal

**Input:** This session uses the processor output (e.g. `output_2017.pkl`) from Session 3. Run `python run_analysis.py` or `run_analysis.py --full` first if you do not have the pickle file.

![Placeholder: schematic limit plot (signal strength vs mediator mass or tanβ)](figures/session4_fig_limits_placeholder.png)

---
## 0. Load processor output

Load the per-dataset histograms and cutflows. Each key in `results` is a dataset name; each value is an accumulator with `met_sr`, `cutflow`, etc.

In [ ]:
# Ensure project root is on sys.path (for SWAN / any kernel)
import sys, os
_script_dir = os.path.dirname(os.path.abspath("."))
if _script_dir:
    sys.path.insert(0, os.path.join(_script_dir, ".."))

# Avoid recursion limit during deep pickle load (accumulator + hist)
try:
    sys.setrecursionlimit(8000)
except Exception:
    pass

import pickle
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import poisson, chi2

# Load processor output (from run_analysis.py; saved to output/)
import os

def _load_results():
    candidates = [
        "output/output_2017.pkl", "output/output_2017_full.pkl",
        "output_2017.pkl", "output_2017_full.pkl",
        "../output/output_2017.pkl", "../output/output_2017_full.pkl",
    ]
    for path in candidates:
        if os.path.isfile(path):
            return pickle.load(open(path, "rb"))
    raise FileNotFoundError(
        "No processor output found. Run: python run_analysis.py (or run_analysis.py --full) first. "
        "Outputs are in output/output_2017.pkl or output/output_2017_full.pkl."
    )

results = _load_results()

all_datasets = list(results.keys())
data_datasets = [d for d in all_datasets if "Run2017" in d or "Single" in d or "MET" in d]
bkg_datasets = [d for d in all_datasets if d not in data_datasets]

# Main observable: cos(theta*) when available, else MET SR
def get_main_hist(h):
    return h.get("cos_theta_star_sr") or h["met_sr"]

def sum_hists(results, names):
    out = None
    for n in names:
        if n not in results:
            continue
        h = get_main_hist(results[n])
        if out is None:
            out = h.copy()
        else:
            out = out + h
    return out

data_hist = sum_hists(results, data_datasets)
bkg_hist = sum_hists(results, bkg_datasets)
if data_hist is not None:
    obs = np.asarray(data_hist.values()).flatten()
    obs = np.maximum(obs, 0)
else:
    obs = None
if bkg_hist is not None:
    bkg = np.asarray(bkg_hist.values()).flatten()
    bkg = np.maximum(bkg, 1e-6)
    if obs is None:
        obs = np.zeros(len(bkg))
else:
    bkg = np.ones(25) * 10
    if obs is None:
        obs = np.zeros(25)

print("Data sum:", obs.sum(), "Bkg sum:", bkg.sum(), "(main observable: cos_theta_star_sr if present else met_sr)")

---
## 1. Weights and systematics

The processor uses **genWeight** (sign only for MC) when filling histograms. In a full analysis we would add scale factors (e.g. pileup, lepton ID) and systematic variations.

**Systematic uncertainties** affect the predicted yields and shapes:
- **Normalisation:** e.g. ±20% per process (theory, luminosity).
- **Shape:** e.g. MET scale, b-tag efficiency, jet energy scale.

Here we introduce a simple **normalisation systematic** per background: we scale each histogram by a factor \((1 + \alpha \cdot \sigma)\), where \(\sigma\) is the relative uncertainty (e.g. 0.2) and \(\alpha\) is a nuisance parameter (fitted or fixed at ±1 for up/down).

In [ ]:
# Build nominal histogram per dataset. Main observable: cos_theta_star_sr (else met_sr).
from hist import Hist, axis

def get_met_sr_hist(results, dataset_name):
    """Return the met_sr histogram for a dataset."""
    if dataset_name not in results:
        return None
    return results[dataset_name]["met_sr"]

def get_main_hist(results, dataset_name):
    """Return cos_theta_star_sr (main observable) if present, else met_sr."""
    if dataset_name not in results:
        return None
    acc = results[dataset_name]
    return acc.get("cos_theta_star_sr") or acc["met_sr"]

# Example: list background and data dataset names (adjust to your results keys)
# Backgrounds: typically MC (ttbar, Zvv, Wjets, etc.); data: MET_Run2017* or Single*
all_datasets = list(results.keys())
data_datasets = [d for d in all_datasets if "Run2017" in d or "Single" in d or "MET" in d]
bkg_datasets = [d for d in all_datasets if d not in data_datasets]
print("Data (example):", data_datasets[:3])
print("Backgrounds (example):", bkg_datasets[:5])

---
## 2. Binned fit

We fit a **binned likelihood**: in each MET bin \(i\), the observed count \(n_i\) is Poisson with mean \(\mu \cdot s_i + \sum_b b_{i,b}\), where \(s_i\) is the signal prediction, \(b_{i,b}\) the background \(b\) in bin \(i\), and \(\mu\) is the signal strength (1 = SM signal, 0 = background only).

**Simplified:** Assume only backgrounds (no signal histogram yet). Fit the sum of background histograms to the data histogram by scaling backgrounds with normalisation factors (or fix them and compute \(\chi^2\)). If you have a signal histogram, add \(\mu \cdot s\) and fit \(\mu\).

In [ ]:
from scipy.optimize import minimize
from scipy.stats import poisson

# Binned fit (one scale factor for total background)

def nll(scale):
    lam = scale * bkg
    return -np.sum(poisson.logpmf(obs.astype(int), np.maximum(lam, 1e-10)))

fit = minimize(nll, x0=[1.0], bounds=[(0.01, 10)])
scale_best = fit.x[0]
expected = scale_best * bkg
print("Best-fit background scale:", scale_best)
print("Total observed:", obs.sum(), "Total expected (after fit):", expected.sum())

---
## 3. Goodness of fit

After the fit, check how well the model describes the data:
- **\(\chi^2\):** \(\chi^2 = \sum_i (n_i - \lambda_i)^2 / \lambda_i\) (or use variance \(\lambda_i\) for Poisson).
- **Degrees of freedom:** number of bins minus number of fitted parameters.
- **p-value:** \(P(\chi^2 \geq \chi^2_{\text{obs}})\) from the \(\chi^2\) distribution.
- **Pulls:** \((n_i - \lambda_i) / \sigma_i\) per bin to spot which bins disagree.

In [ ]:
from scipy.stats import chi2

# Goodness of fit using chi2 and pulls
chi2_val = np.sum((obs - expected)**2 / np.where(expected > 0, expected, 1))
ndof = len(obs) - 1
pvalue = 1 - chi2.cdf(chi2_val, ndof)
print(f"chi2 = {chi2_val:.2f}, ndof = {ndof}, p-value = {pvalue:.4f}")

pulls = (obs - expected) / np.sqrt(np.where(expected > 0, expected, 1))
plt.figure(figsize=(8, 3))
plt.bar(range(len(pulls)), pulls, edgecolor="black", alpha=0.7)
plt.xlabel("Bin")
plt.ylabel("Pull")
plt.title("Pull distribution")
plt.axhline(0, color="gray", ls="--")
plt.tight_layout()
plt.show()

---
## 4. Limit calculation

An **upper limit** on the signal strength \(\mu\) at e.g. 95% CL says: we exclude \(\mu > \mu_{\text{limit}}\) at 95% confidence. Common methods:
- **Asymptotic formula:** Use the profile likelihood ratio and the asymptotic distribution (e.g. \(2\Delta\ln L\) is chi-squared distributed).
- **pyhf:** The [pyhf](https://pyhf.readthedocs.io/) library implements histogram fits and asymptotic limits (and toys if needed).

**Simplified exercise:** For a single bin (e.g. total SR yield), the 95% CL upper limit on the number of signal events can be computed with the one-sided Poisson formula (e.g. \(n_{\text{obs}}\) observed, \(b\) expected background → upper limit on \(s\)). Or use scipy to find \(\mu\) such that the profile likelihood (or a simple likelihood) gives the desired CL.

In [ ]:
# Simple 95% CL upper limit (single bin: total SR yield)

# One-bin: observed = obs.sum(), background = expected.sum(); upper limit on signal s at 95% CL
n_obs = int(obs.sum())
b = expected.sum()

# Simple approximate: s_95 such that P(n <= n_obs | b + s_95) = 0.05 (one-sided)
from scipy.stats import poisson

def p_exceed(s):
    return 1 - poisson.cdf(n_obs, b + s)

s_vals = np.linspace(0, 50, 200)
p_vals = [p_exceed(s) for s in s_vals]
idx = np.argmin(np.abs(np.array(p_vals) - 0.05))
s_95 = s_vals[idx]
print(f"Approximate 95% CL upper limit on signal events (single bin): s_95 ≈ {s_95:.1f}")

---
## 5. Summary

- **Weights:** MC event weights (e.g. genWeight) are used when filling histograms; systematics are encoded as normalisation or shape variations.
- **Fit:** Binned Poisson likelihood fit of signal + backgrounds to data; minimisation gives best-fit parameters (e.g. signal strength \(\mu\)).
- **Goodness of fit:** \(\chi^2\), p-value, and pulls quantify how well the model describes the data.
- **Limit:** Upper limit on signal (e.g. 95% CL) via asymptotic formula or pyhf.

**Further reading:** pyhf documentation, TRExFitter, Combine (CMS).

### Exercises

1. Add a second systematic (e.g. shape: scale MET by 1.02 for "up") and re-fit.
2. Compute the limit for a different signal hypothesis (e.g. different cross-section).
3. (If time) Compare asymptotic limit with a toy Monte Carlo limit.